Выполнил: __Кондрат Никита Игоревич__, группа: __P3123__

# Лабораторная работа 8. Скрапинг

##  Цель работы
Освоение методов скрапинга данных из веб-страниц.

## Теоретические сведения
Основные понятия:

Скрейпинг / Веб-скрейпинг — это технология получения веб-данных путём извлечения их со страниц веб-ресурсов.


Объект для скрапинга: https://news.itmo.ru/ru.

Необходимо сохранить полученные данных в формате csv внутри колаба.

Структура данных :

1. Идентификатор новости (целочисленное число из URL)
2. Название новости
3. Дата её размещения
3. URL на страницу с конкретной новостью.

In [8]:
import csv
import requests
from bs4 import BeautifulSoup
import time

In [9]:
DOMAIN = 'https://news.itmo.ru'
SEARCH_DOMAIN = 'https://news.itmo.ru/ru/search/?search='

In [10]:
def csv_writer(data, path):
    """
    Write data to a CSV file path
    """
    with open(path, "a", newline='') as csv_file:
        writer = csv.writer(csv_file, delimiter=',')
        for line in data:
            writer.writerow(line)

In [11]:
headers = []
urls = []

queries = [
    'нейротехнологии',
    'нейротехнологии и программирование',
    'ИИ',
    'Машинное обучение',
    'Искусственный интеллект',
    'Нейросети'
]

# div.weeklyevents > h2 > span - text
# li.weeklyevent > h4 > a - text

"""
<li class="weeklyevent"><h4><a href="/ru/education/cooperation/news/14792/">Более 400 участников, 30 регионов и 6 треков: главные итоги и цифры рекордного сезона финалов НТО в ИТМО</a></h4><p>… Native»
	«Компьютерные системы и технологии»
	«Разработка программного обеспечения»
	«Инженерия искусственного интеллекта»
	«Языковые модели и искусственный интеллект»
	«<span class="selection">Нейротехнологии и программирование</span>»
	«Системное и прикладное программное обеспечение»
|s=2,center|
«Летающая робототехника» ― 49 финалистов. Партнеры ― «Клевер COEX», «Свеза» и Акаде…</p><p>06.04.2026</p></li>
"""


for query in queries:
  data = requests.get(SEARCH_DOMAIN+query)
  time.sleep(1)
  bs = BeautifulSoup(data.text, 'html.parser')
  total_news = bs.find_all('div', {'class': 'weeklyevents'})
  total_news_count = int(total_news[0].find('h2').find('span').get_text().split()[0])

  news_headers = bs.find_all('li', {'class': 'weeklyevent'})
  for _n in news_headers:
    post = _n.find('h4').find('a').get_text()
    post_url = _n.find('h4').find('a')['href']
    if (DOMAIN+post) not in urls:
        urls.append(DOMAIN+post_url)
        headers.append(post)

print(len(urls))


50


In [12]:
print(headers[:2])
print(urls[:2])

['Более 400 участников, 30 регионов и 6 треков: главные итоги и цифры рекордного сезона финалов НТО в ИТМО', 'Топ образование по программированию и ИИ в 2026-м: на что смотреть и как выбирать']
['https://news.itmo.ru/ru/education/cooperation/news/14792/', 'https://news.itmo.ru/ru/education/trend/news/14771/']


In [13]:
import csv
import requests
from bs4 import BeautifulSoup
import re # для извлечения ID из URL
import os

# Списки `headers` и `urls` уже заполнены из предыдущих ячеек.

news_data = []
basic_csv_file = 'scraped_news_data.csv'
detailed_folder = 'news_content'
detailed_csv_file = os.path.join(detailed_folder, 'detailed_news.csv')

os.makedirs(detailed_folder, exist_ok=True)

print("Начинается сбор даты для каждой новости. Это может занять некоторое время...")

for i, (header, url) in enumerate(zip(headers, urls)):
    news_id = 'N/A'
    news_date = 'Дата не найдена'
    views = '0'
    text = ''
    tags = []

    # Извлечение ID из URL
    match = re.search(r'/(\d+)/?$', url)
    if match:
        news_id = match.group(1)

    try:
        response = requests.get(url)
        response.raise_for_status() # Вызывает исключение для HTTP ошибок
        soup_news = BeautifulSoup(response.text, 'html.parser')

        # Поиск даты размещения новости на странице
        # На основе ручного анализа, дата находится в <div class="time"><time>...
        date_element = soup_news.find('time', datetime=True)
        if date_element:
            news_date = date_element.get('datetime', 'Дата не найдена').split('T')[0]
        # else:
        #    print(f"Предупреждение: Дата не найдена для {url}")

        views_el = soup_news.find('span', class_='icon eye')
        if views_el:
            views = views_el.get_text(strip=True)

        content_block = soup_news.find('div', class_='content')
        if content_block:
            elements = content_block.find_all(['p', 'li', 'blockquote', 'h2', 'h3', 'h4'])
            text = ' '.join([el.get_text(strip=True) for el in elements if el.get_text(strip=True)])
            text = re.sub(r'\s+', ' ', text).strip()

        tags_block = soup_news.find('ul', class_='tags')
        if tags_block:
            tags = [tag.get_text(strip=True) for tag in tags_block.find_all('a') if tag.get_text(strip=True)]

        text = re.sub(r'\s+', ' ', text).strip()

    except requests.exceptions.RequestException as e:
        print(f"Ошибка при запросе {url}: {e}")
    except Exception as e:
        print(f"Ошибка при парсинге {url}: {e}")

    news_data.append({
        'Идентификатор новости': news_id,
        'Название новости': header,
        'Дата её размещения': news_date,
        'URL на страницу с конкретной новостью': url,
        'Количество просмотров': views,
        'Текст': text,
        'Теги': ', '.join(tags)
    })

    if (i + 1) % 10 == 0:
        print(f"Обработано {i + 1} новостей...")

print("Сбор данных завершен. Начинается запись в CSV.")

basic_fieldnames = [
    'Идентификатор новости',
    'Название новости',
    'Дата её размещения',
    'URL на страницу с конкретной новостью'
]
try:
    with open(basic_csv_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=basic_fieldnames)
        writer.writeheader()
        for row in news_data:
            writer.writerow({k: row[k] for k in basic_fieldnames})
    print(f"Данные успешно записаны в файл {basic_csv_file}")
except IOError as e:
    print(f"Ошибка при записи в файл {basic_csv_file}: {e}")

detailed_fieldnames = [
    'Идентификатор',
    'Название новости',
    'Дата размещения',
    'Количество просмотров',
    'Текст',
    'Теги'
]

detailed_news_data = [{
    'Идентификатор': row['Идентификатор новости'],
    'Название новости': row['Название новости'],
    'Дата размещения': row['Дата её размещения'],
    'Количество просмотров': row['Количество просмотров'],
    'Текст': row['Текст'],
    'Теги': row['Теги']
} for row in news_data]

try:
    with open(detailed_csv_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.DictWriter(csvfile, fieldnames=detailed_fieldnames)
        writer.writeheader()
        writer.writerows(detailed_news_data)
    print(f"Данные успешно записаны в файл {detailed_csv_file}")
except IOError as e:
    print(f"Ошибка при записи в файл {detailed_csv_file}: {e}")


Начинается сбор даты для каждой новости. Это может занять некоторое время...
Обработано 10 новостей...
Обработано 20 новостей...
Обработано 30 новостей...
Обработано 40 новостей...
Обработано 50 новостей...
Сбор данных завершен. Начинается запись в CSV.
Данные успешно записаны в файл scraped_news_data.csv
Данные успешно записаны в файл news_content/detailed_news.csv
